In [6]:
!pip install scikit-learn pandas numpy matplotlib joblib -q

In [7]:
import pandas as pd

url_80 = "https://raw.githubusercontent.com/HimanshuKhale/ipl_sentiment/main/ipl_sentiment_small_dataset.csv"

df = pd.read_csv(url_80)

print(df.shape)
df.head()

(80, 4)


,post_id,statement,subject_keyword,sentiment
0,1,"RCB finally played like champions today, the b...",RCB,positive
1,2,RCB fans deserve better than another middle or...,RCB,negative
2,3,RCB's batting was decent but the fielding stil...,RCB,neutral
3,4,Virat's energy makes RCB matches feel electric...,RCB,positive
4,5,"RCB again lost control in the death overs, sam...",RCB,negative


In [8]:
print("Columns in dataset:")
print(df.columns.tolist())

# Automatically detect text column
text_col = None

for col in ["tweet", "statement", "text", "comment"]:
    if col in df.columns:
        text_col = col
        break

if text_col is None:
    raise ValueError(
        f"No text column found. Available columns: {df.columns.tolist()}"
    )

df["model_text"] = (
    df["subject_keyword"].astype(str)
    + " [SEP] "
    + df[text_col].astype(str)
)

df[["model_text"]].head()

Columns in dataset:
['post_id', 'statement', 'subject_keyword', 'sentiment']


,model_text
0,RCB [SEP] RCB finally played like champions to...
1,RCB [SEP] RCB fans deserve better than another...
2,RCB [SEP] RCB's batting was decent but the fie...
3,RCB [SEP] Virat's energy makes RCB matches fee...
4,RCB [SEP] RCB again lost control in the death ...


In [9]:
from sklearn.model_selection import train_test_split

X = df["model_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
    HashingVectorizer
)

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier

In [11]:
models = {

    "Count_NB":
    Pipeline([
        ("vec", CountVectorizer()),
        ("clf", MultinomialNB())
    ]),

    "TFIDF_LR":
    Pipeline([
        ("vec", TfidfVectorizer(
            ngram_range=(1,2)
        )),
        ("clf", LogisticRegression(
            max_iter=3000
        ))
    ]),

    "TFIDF_SVC":
    Pipeline([
        ("vec", TfidfVectorizer(
            ngram_range=(1,2)
        )),
        ("clf", LinearSVC())
    ]),

    "CHAR_SVC":
    Pipeline([
        ("vec",
         TfidfVectorizer(
             analyzer="char",
             ngram_range=(3,5)
         )),
        ("clf", LinearSVC())
    ]),

    "HASH_SGD":
    Pipeline([
        ("vec",
         HashingVectorizer(
             n_features=2**16
         )),
        ("clf",
         SGDClassifier())
    ])
}

In [12]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

results = []

best_model = None
best_f1 = 0

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)

    f1 = f1_score(
        y_test,
        pred,
        average="macro"
    )

    results.append(
        [name, acc, f1]
    )

    if f1 > best_f1:
        best_f1 = f1
        best_model = model

results_df = pd.DataFrame(
    results,
    columns=["Model",
             "Accuracy",
             "Macro_F1"]
)

results_df.sort_values(
    "Macro_F1",
    ascending=False
)

,Model,Accuracy,Macro_F1
0,Count_NB,0.6875,0.673401
3,CHAR_SVC,0.6875,0.665501
2,TFIDF_SVC,0.6250,0.640476
4,HASH_SGD,0.5625,0.561254
1,TFIDF_LR,0.4375,0.454545


In [13]:
import joblib

joblib.dump(
    best_model,
    "best_model.pkl"
)

['best_model.pkl']

In [14]:
preds = best_model.predict(
    df["model_text"]
)

submission = pd.DataFrame({
    "post_id": df["post_id"],
    "prediction": preds
})

submission.to_csv(
    "results_80R.csv",
    index=False
)

submission.head()

,post_id,prediction
0,1,positive
1,2,negative
2,3,neutral
3,4,negative
4,5,negative


In [15]:
from google.colab import files

files.download(
    "results_80R.csv"
)

files.download(
    "best_model.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
results_df.sort_values(
    "Macro_F1",
    ascending=False
)

,Model,Accuracy,Macro_F1
0,Count_NB,0.6875,0.673401
3,CHAR_SVC,0.6875,0.665501
2,TFIDF_SVC,0.6250,0.640476
4,HASH_SGD,0.5625,0.561254
1,TFIDF_LR,0.4375,0.454545
